# Insurance Premium Prediction (100k Dataset)
## GPU-Accelerated Model Training — Hard Revision

**Target Variable:** `annual_premium`
**Dataset:** 100,000 medical insurance records
**Models:** Linear Regression, Random Forest (GPU), XGBoost (GPU), LightGBM
**Approach:** Comprehensive feature engineering with no data leakage


# 1. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import joblib
from scipy import stats
import warnings
warnings.simplefilter(action='ignore')

print("All imports loaded successfully.")


# 2. Data Loading


In [ ]:
# Install Kaggle API client
!pip install -q kaggle

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d mohankrishnathalla/medical-insurance-cost-prediction -p /content

# Unzip the dataset
!unzip -o /content/medical-insurance-cost-prediction.zip -d /content

# Load data
data = pd.read_csv('/content/medical_insurance.csv')

# Drop person_id (unique identifier, not a feature)
if "person_id" in data.columns:
    data.drop(columns=["person_id"], inplace=True)

print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")


# 3. Initial Exploration


In [ ]:
print(f"Shape: {data.shape}")
print(f"\nData types:")
print(data.dtypes.value_counts())
print(f"\nMissing values: {data.isnull().sum().sum()}")
data.head()


In [ ]:
data.describe()


In [ ]:
# Check for missing values per column
missing = data.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("Columns with missing values:")
    print(missing.sort_values(ascending=False))
else:
    print("No missing values found.")


# 4. Missing Value Handling


In [ ]:
# Fill missing categorical values
if 'alcohol_freq' in data.columns:
    data['alcohol_freq'] = data['alcohol_freq'].fillna('Unknown')

# Fill missing smoking status
if 'smoker' in data.columns and data['smoker'].isnull().sum() > 0:
    data['smoker'] = data['smoker'].fillna('Unknown')

print(f"Missing values after handling: {data.isnull().sum().sum()}")


# 5. Target Distribution & Outlier Removal


## 5.1 Target: Annual Premium


In [ ]:
# Our target is annual_premium (not annual_medical_cost)
# Insurance premiums are legitimately calculated from patient demographics,
# health status, and risk factors — making this a non-leaky, high-signal target.

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(data['annual_premium'], kde=True, bins=50, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Annual Premium', fontsize=13)
axes[0].set_xlabel('Annual Premium ($)')
axes[0].axvline(data['annual_premium'].mean(), color='red', linestyle='--',
                label=f"Mean: {data['annual_premium'].mean():.2f}")
axes[0].axvline(data['annual_premium'].median(), color='green', linestyle='--',
                label=f"Median: {data['annual_premium'].median():.2f}")
axes[0].legend()

sns.boxplot(y=data['annual_premium'], color='steelblue', ax=axes[1])
axes[1].set_title('Box Plot of Annual Premium', fontsize=13)

plt.tight_layout()
plt.show()

print(f"Mean: ${data['annual_premium'].mean():.2f}")
print(f"Median: ${data['annual_premium'].median():.2f}")
print(f"Std: ${data['annual_premium'].std():.2f}")


## 5.2 Key Feature Relationships


In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
sns.boxplot(x='smoker', y='annual_premium', data=data)
plt.title('Annual Premium by Smoker Status')

plt.subplot(2, 2, 2)
sns.boxplot(x='region', y='annual_premium', data=data)
plt.title('Annual Premium by Region')

plt.subplot(2, 2, 3)
sns.scatterplot(x='age', y='annual_premium', data=data, alpha=0.1, s=5)
plt.title('Annual Premium vs Age')

plt.subplot(2, 2, 4)
sns.scatterplot(x='bmi', y='annual_premium', data=data, alpha=0.1, s=5)
plt.title('Annual Premium vs BMI')

plt.tight_layout()
plt.show()


## 5.3 Outlier Removal


In [ ]:
# Remove outliers using z-score method (threshold = 3)
data_cleaned = data[np.abs(stats.zscore(data['annual_premium'])) < 3]

rows_removed = len(data) - len(data_cleaned)
print(f"Rows before:   {len(data):,}")
print(f"Rows after:    {len(data_cleaned):,}")
print(f"Rows removed:  {rows_removed:,} ({rows_removed/len(data)*100:.1f}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(data['annual_premium'], kde=True, bins=50, color='salmon', ax=axes[0])
axes[0].set_title('BEFORE Outlier Removal', fontsize=13)
axes[0].set_xlabel('Annual Premium ($)')

sns.histplot(data_cleaned['annual_premium'], kde=True, bins=50, color='seagreen', ax=axes[1])
axes[1].set_title('AFTER Outlier Removal', fontsize=13)
axes[1].set_xlabel('Annual Premium ($)')

plt.suptitle('Effect of Outlier Removal on Target Distribution', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


# 5.4 Correlation with Target


In [ ]:
# Encode categorical columns for correlation
data_corr = data_cleaned.copy()
label_encoder = LabelEncoder()
for col in data_corr.select_dtypes(include='object').columns:
    data_corr[col] = label_encoder.fit_transform(data_corr[col])

corr_matrix = data_corr.corr()

# Show top 15 features most correlated with annual_premium
target_corr = corr_matrix['annual_premium'].drop('annual_premium').abs().sort_values(ascending=False)
top_features = target_corr.head(15).index.tolist() + ['annual_premium']

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix.loc[top_features, top_features], annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, linecolor='white')
plt.title('Top 15 Features Correlated with Annual Premium', fontsize=14)
plt.tight_layout()
plt.show()

print("\nTop 15 correlations with annual_premium:")
print(target_corr.head(15).to_string())


# 6. Feature Engineering & Encoding


In [ ]:
# === Comprehensive Feature Engineering ===
# Inspired by high-performance reference analysis

# 1. HEALTHCARE UTILIZATION FEATURES
data_cleaned['total_conditions'] = (
    data_cleaned['hypertension'] + data_cleaned['diabetes'] +
    data_cleaned['asthma'] + data_cleaned['copd'] +
    data_cleaned['cardiovascular_disease'] + data_cleaned['cancer_history'] +
    data_cleaned['kidney_disease'] + data_cleaned['liver_disease'] +
    data_cleaned['arthritis'] + data_cleaned['mental_health']
)

proc_cols = ['proc_imaging_count', 'proc_surgery_count', 'proc_physio_count',
             'proc_consult_count', 'proc_lab_count']
existing_proc = [c for c in proc_cols if c in data_cleaned.columns]
if existing_proc:
    data_cleaned['total_procedures'] = data_cleaned[existing_proc].sum(axis=1)

data_cleaned['utilization_score'] = (
    data_cleaned['visits_last_year'] +
    data_cleaned['hospitalizations_last_3yrs'] * 10 +
    data_cleaned['medication_count'] * 2
)
print("Created healthcare utilization features")

# 2. HEALTH RISK COMPOSITE SCORES
data_cleaned['complex_patient'] = (data_cleaned['chronic_count'] >= 3).astype(int)
data_cleaned['high_risk_patient'] = (data_cleaned['risk_score'] > data_cleaned['risk_score'].quantile(0.75)).astype(int)

critical_conds = ['cancer_history', 'kidney_disease', 'liver_disease']
existing_critical = [c for c in critical_conds if c in data_cleaned.columns]
if existing_critical:
    data_cleaned['critical_conditions'] = data_cleaned[existing_critical].sum(axis=1)

cardio_conds = ['hypertension', 'diabetes', 'cardiovascular_disease']
existing_cardio = [c for c in cardio_conds if c in data_cleaned.columns]
if existing_cardio:
    data_cleaned['cardiovascular_risk'] = data_cleaned[existing_cardio].sum(axis=1).clip(upper=3)
print("Created health risk composite scores")

# 3. AGE-HEALTH INTERACTION FEATURES
data_cleaned['age_chronic_interaction'] = data_cleaned['age'] * data_cleaned['chronic_count']
data_cleaned['age_risk_score'] = data_cleaned['age'] * data_cleaned['risk_score']
data_cleaned['age_x_bmi'] = data_cleaned['age'] * data_cleaned['bmi']
data_cleaned['bmi_x_chronic'] = data_cleaned['bmi'] * data_cleaned['chronic_count']
data_cleaned['risk_x_chronic'] = data_cleaned['risk_score'] * data_cleaned['chronic_count']
data_cleaned['elderly_chronic'] = ((data_cleaned['age'] > 65) & (data_cleaned['chronic_count'] >= 2)).astype(int)
data_cleaned['young_high_risk'] = ((data_cleaned['age'] < 40) & (data_cleaned['chronic_count'] >= 1)).astype(int)
print("Created age-health interaction features")

# 4. INSURANCE & FINANCIAL FEATURES
data_cleaned['deductible_to_income'] = data_cleaned['deductible'] / (data_cleaned['income'] + 1)
data_cleaned['out_of_pocket_risk'] = data_cleaned['deductible'] + data_cleaned['copay']
data_cleaned['coverage_adequacy'] = data_cleaned['income'] / (data_cleaned['deductible'] + data_cleaned['copay'] + 1)
print("Created insurance & financial features")

# 5. LIFESTYLE RISK FEATURES
data_cleaned['is_obese'] = (data_cleaned['bmi'] >= 30).astype(int)
data_cleaned['is_overweight'] = (data_cleaned['bmi'] >= 25).astype(int)

lifestyle_risk = data_cleaned['is_obese'].astype(int) * 2
if 'exercise_frequency' in data_cleaned.columns:
    lifestyle_risk = lifestyle_risk + (7 - data_cleaned['exercise_frequency']).clip(lower=0)
data_cleaned['lifestyle_risk_score'] = lifestyle_risk
print("Created lifestyle risk features")

# 6. CLINICAL METRICS
data_cleaned['bp_ratio'] = data_cleaned['systolic_bp'] / (data_cleaned['diastolic_bp'] + 1)
data_cleaned['pulse_pressure'] = data_cleaned['systolic_bp'] - data_cleaned['diastolic_bp']
data_cleaned['hypertensive_flag'] = ((data_cleaned['systolic_bp'] > 140) | (data_cleaned['diastolic_bp'] > 90)).astype(int)
data_cleaned['high_cholesterol'] = (data_cleaned['ldl'] > 130).astype(int)
if 'hba1c' in data_cleaned.columns:
    data_cleaned['diabetic_control'] = (data_cleaned['hba1c'] > 6.5).astype(int)
print("Created clinical metrics features")

# 7. CLAIMS HISTORY FEATURES (legitimate for predicting premiums)
data_cleaned['avg_claim_size'] = data_cleaned['total_claims_paid'] / (data_cleaned['claims_count'] + 1)
data_cleaned['claims_frequency'] = data_cleaned['claims_count'] / 12  # per month
data_cleaned['claims_x_chronic'] = data_cleaned['total_claims_paid'] * data_cleaned['chronic_count']
data_cleaned['claims_x_risk'] = data_cleaned['total_claims_paid'] * data_cleaned['risk_score']
print("Created claims history features")

# 8. POLYNOMIAL FEATURES
data_cleaned['age_sq'] = data_cleaned['age'] ** 2
data_cleaned['bmi_sq'] = data_cleaned['bmi'] ** 2
data_cleaned['chronic_count_sq'] = data_cleaned['chronic_count'] ** 2
data_cleaned['risk_score_sq'] = data_cleaned['risk_score'] ** 2
print("Created polynomial features")

# 9. LOG TRANSFORMS (for skewed features)
data_cleaned['log_income'] = np.log1p(data_cleaned['income'])
data_cleaned['log_total_claims'] = np.log1p(data_cleaned['total_claims_paid'])
data_cleaned['log_avg_claim'] = np.log1p(data_cleaned['avg_claim_amount'])
data_cleaned['log_days_hosp'] = np.log1p(data_cleaned['days_hospitalized_last_3yrs'])
print("Created log-transform features")

print(f"\nShape after feature engineering: {data_cleaned.shape}")


In [ ]:
# One-hot encode categorical features
dum = pd.get_dummies(data_cleaned.select_dtypes(include='object'), drop_first=True, dtype=int)
print(f"Encoded features shape: {dum.shape}")

# Combine numeric + encoded features
data_model = pd.concat([data_cleaned.select_dtypes(exclude='object'), dum], axis=1)
print(f"Final feature matrix shape: {data_model.shape}")


In [ ]:
# === Define X and y ===
# Target: annual_premium
# Drop: monthly_premium (derived: annual_premium / 12), annual_medical_cost (avoid reverse leakage)
drop_cols = ['annual_premium', 'monthly_premium', 'annual_medical_cost']
X = data_model.drop(columns=drop_cols, errors='ignore')
y_orig = data_model['annual_premium']

# Log-transform the target for better model performance
y = np.log1p(y_orig)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures dropped: {drop_cols}")
print(f"Number of features: {X.shape[1]}")
print(f"\nTarget (original): mean=${y_orig.mean():.2f}, median=${y_orig.median():.2f}")
print(f"Target (log):      mean={y.mean():.4f}, median={y.median():.4f}")


# 7. Model Training


## 7.1 GPU-Accelerated Imports


In [ ]:
# GPU-accelerated models
try:
    from cuml.ensemble import RandomForestRegressor as RFR
    print("Using cuML GPU-accelerated RandomForestRegressor")
except ImportError:
    from sklearn.ensemble import RandomForestRegressor as RFR
    print("cuML not available, using sklearn RandomForestRegressor (CPU)")

from xgboost import XGBRegressor

try:
    from lightgbm import LGBMRegressor
    has_lgbm = True
    print("LightGBM available")
except ImportError:
    has_lgbm = False
    print("LightGBM not available - install with: !pip install lightgbm")

print("XGBRegressor ready (GPU via tree_method='hist', device='cuda')")


## 7.2 Data Splits: 70/30, 80/20, 90/10


In [ ]:
# Splits with log-transformed target
splits = {
    '70/30': train_test_split(X, y, test_size=0.30, random_state=7),
    '80/20': train_test_split(X, y, test_size=0.20, random_state=7),
    '90/10': train_test_split(X, y, test_size=0.10, random_state=7),
}

# Keep original y for evaluation in dollar scale
splits_y_orig = {
    '70/30': train_test_split(X, y_orig, test_size=0.30, random_state=7),
    '80/20': train_test_split(X, y_orig, test_size=0.20, random_state=7),
    '90/10': train_test_split(X, y_orig, test_size=0.10, random_state=7),
}

for name, (X_tr, X_te, _, _) in splits.items():
    print(f"  {name}: train={X_tr.shape[0]:,}, test={X_te.shape[0]:,}, features={X_tr.shape[1]}")


## 7.3 Hyperparameter Grids


In [ ]:
# Linear Regression
lr_params = [
    {'fit_intercept': True},
    {'fit_intercept': False},
]

# Random Forest - aggressive tuning
rfr_params = [
    {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 7},
    {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 1, 'random_state': 7},
    {'n_estimators': 700, 'max_depth': 25, 'min_samples_split': 2, 'min_samples_leaf': 1, 'random_state': 7},
]

# XGBoost - aggressive tuning
gbr_params = [
    {'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 3, 'reg_lambda': 1.5, 'random_state': 7},
    {'n_estimators': 1000, 'learning_rate': 0.01, 'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 5, 'reg_lambda': 2.0, 'random_state': 7},
    {'n_estimators': 800, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 1, 'reg_lambda': 1.0, 'random_state': 7},
]

# LightGBM
lgbm_params = [
    {'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 8, 'num_leaves': 63, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.01, 'reg_lambda': 1.0, 'random_state': 7, 'verbose': -1},
    {'n_estimators': 1000, 'learning_rate': 0.01, 'max_depth': 10, 'num_leaves': 127, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_samples': 10, 'reg_alpha': 0.1, 'reg_lambda': 2.0, 'random_state': 7, 'verbose': -1},
    {'n_estimators': 800, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_samples': 30, 'reg_alpha': 0, 'reg_lambda': 1.0, 'random_state': 7, 'verbose': -1},
]

print('Parameter grids ready.')
print(f'  Linear Regression: {len(lr_params)} configs')
print(f'  Random Forest: {len(rfr_params)} configs')
print(f'  XGBoost: {len(gbr_params)} configs')
if has_lgbm:
    print(f'  LightGBM: {len(lgbm_params)} configs')


## 7.4 Training Loop


In [ ]:
# Helper: train on log(y), evaluate in original dollar scale
def evaluate_model(model, X_tr, X_te, split_name):
    _, _, y_tr_orig, y_te_orig = splits_y_orig[split_name]

    y_pred_log_te = model.predict(X_te)
    y_pred_log_tr = model.predict(X_tr)

    y_pred_te = np.expm1(y_pred_log_te)
    y_pred_tr = np.expm1(y_pred_log_tr)

    return {
        'train_r2': r2_score(y_tr_orig, y_pred_tr),
        'test_r2':  r2_score(y_te_orig, y_pred_te),
        'test_mae': mean_absolute_error(y_te_orig, y_pred_te),
        'test_rmse': mean_squared_error(y_te_orig, y_pred_te, squared=False),
    }

best_results = {}
split_names = ['70/30', '80/20', '90/10']

for split_name in split_names:
    best_results[split_name] = {}
    X_tr, X_te, y_tr, y_te = splits[split_name]

    print(f'\n{"="*60}')
    print(f'Split: {split_name}')
    print(f'{"="*60}')

    # -- Linear Regression --
    best_lr_score, best_lr = -999, None
    for p in lr_params:
        m = LinearRegression(**p).fit(X_tr, y_tr)
        metrics = evaluate_model(m, X_tr, X_te, split_name)
        if metrics['test_r2'] > best_lr_score:
            best_lr_score = metrics['test_r2']
            best_lr = (m, p, metrics)
    m, p, metrics = best_lr
    best_results[split_name]['Linear Regression'] = {**metrics, 'params': p, 'model': m}
    print(f'  LR   R2={metrics["test_r2"]:.4f}  MAE={metrics["test_mae"]:.2f}  RMSE={metrics["test_rmse"]:.2f}')

    # -- Random Forest --
    best_rfr_score, best_rfr = -999, None
    for p in rfr_params:
        m = RFR(**p).fit(X_tr, y_tr)
        metrics = evaluate_model(m, X_tr, X_te, split_name)
        if metrics['test_r2'] > best_rfr_score:
            best_rfr_score = metrics['test_r2']
            best_rfr = (m, p, metrics)
    m, p, metrics = best_rfr
    best_results[split_name]['Random Forest'] = {**metrics, 'params': p, 'model': m}
    print(f'  RFR  R2={metrics["test_r2"]:.4f}  MAE={metrics["test_mae"]:.2f}  RMSE={metrics["test_rmse"]:.2f}')

    # -- XGBoost --
    best_gbr_score, best_gbr = -999, None
    for p in gbr_params:
        m = XGBRegressor(tree_method="hist", device="cuda", **p).fit(X_tr, y_tr)
        metrics = evaluate_model(m, X_tr, X_te, split_name)
        if metrics['test_r2'] > best_gbr_score:
            best_gbr_score = metrics['test_r2']
            best_gbr = (m, p, metrics)
    m, p, metrics = best_gbr
    best_results[split_name]['XGBoost'] = {**metrics, 'params': p, 'model': m}
    print(f'  XGB  R2={metrics["test_r2"]:.4f}  MAE={metrics["test_mae"]:.2f}  RMSE={metrics["test_rmse"]:.2f}')

    # -- LightGBM --
    if has_lgbm:
        best_lgbm_score, best_lgbm = -999, None
        for p in lgbm_params:
            m = LGBMRegressor(**p).fit(X_tr, y_tr)
            metrics = evaluate_model(m, X_tr, X_te, split_name)
            if metrics['test_r2'] > best_lgbm_score:
                best_lgbm_score = metrics['test_r2']
                best_lgbm = (m, p, metrics)
        m, p, metrics = best_lgbm
        best_results[split_name]['LightGBM'] = {**metrics, 'params': p, 'model': m}
        print(f'  LGBM R2={metrics["test_r2"]:.4f}  MAE={metrics["test_mae"]:.2f}  RMSE={metrics["test_rmse"]:.2f}')

print('\nTraining complete.')


# 8. Results & Evaluation


## 8.1 Comparison Matrices


In [ ]:
model_names = list(best_results['70/30'].keys())
split_names = ['70/30', '80/20', '90/10']

matrix_test = pd.DataFrame(index=model_names, columns=split_names, dtype=float)
matrix_train = pd.DataFrame(index=model_names, columns=split_names, dtype=float)
matrix_mae = pd.DataFrame(index=model_names, columns=split_names, dtype=float)
matrix_rmse = pd.DataFrame(index=model_names, columns=split_names, dtype=float)

for sp in split_names:
    for mn in best_results[sp]:
        r = best_results[sp][mn]
        matrix_test.loc[mn, sp] = round(r['test_r2'], 4)
        matrix_train.loc[mn, sp] = round(r['train_r2'], 4)
        matrix_mae.loc[mn, sp] = round(r['test_mae'], 2)
        matrix_rmse.loc[mn, sp] = round(r['test_rmse'], 2)

print('=== Test R-squared Matrix ===')
display(matrix_test)
print('\n=== Test MAE Matrix ===')
display(matrix_mae)
print('\n=== Test RMSE Matrix ===')
display(matrix_rmse)


## 8.2 Performance Heatmaps


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.heatmap(matrix_test.astype(float), annot=True, fmt='.4f',
            cmap='YlGnBu', vmin=0, vmax=1, ax=axes[0, 0],
            linewidths=0.5, linecolor='white')
axes[0, 0].set_title('Test R-squared', fontsize=13)

sns.heatmap(matrix_train.astype(float), annot=True, fmt='.4f',
            cmap='YlOrRd', vmin=0, vmax=1, ax=axes[0, 1],
            linewidths=0.5, linecolor='white')
axes[0, 1].set_title('Train R-squared', fontsize=13)

sns.heatmap(matrix_mae.astype(float), annot=True, fmt='.2f',
            cmap='Blues', ax=axes[1, 0],
            linewidths=0.5, linecolor='white')
axes[1, 0].set_title('Test MAE (lower is better)', fontsize=13)

sns.heatmap(matrix_rmse.astype(float), annot=True, fmt='.2f',
            cmap='Reds', ax=axes[1, 1],
            linewidths=0.5, linecolor='white')
axes[1, 1].set_title('Test RMSE (lower is better)', fontsize=13)

for ax in axes.flat:
    ax.set_xlabel('Data Split')
    ax.set_ylabel('Model')

plt.suptitle('Model Performance Comparison — Annual Premium Prediction', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


## 8.3 Detailed Summary


In [ ]:
rows = []
for sp in split_names:
    for mn in best_results[sp]:
        r = best_results[sp][mn]
        rows.append({
            'Split': sp, 'Model': mn,
            'Best Params': str(r['params']),
            'Train R2': round(r['train_r2'], 4),
            'Test R2':  round(r['test_r2'],  4),
            'Test MAE': round(r['test_mae'], 2),
            'Test RMSE': round(r['test_rmse'], 2),
        })
summary_df = pd.DataFrame(rows)
display(summary_df)


# 9. Model Diagnostics


## 9.1 Predicted vs Actual (Best Model per Split)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, split_name in enumerate(split_names):
    models = best_results[split_name]
    best_model_name = max(models, key=lambda k: models[k]['test_r2'])
    best = models[best_model_name]

    X_tr, X_te, _, _ = splits[split_name]
    _, _, _, y_te_orig = splits_y_orig[split_name]

    y_pred_log = best['model'].predict(X_te)
    y_pred = np.expm1(y_pred_log)

    axes[idx].scatter(y_te_orig, y_pred, alpha=0.2, s=5, color='steelblue')
    min_val = min(y_te_orig.min(), y_pred.min())
    max_val = max(y_te_orig.max(), y_pred.max())
    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
    axes[idx].set_title(f'{split_name}: {best_model_name}\nR2={best["test_r2"]:.4f}', fontsize=12)
    axes[idx].set_xlabel('Actual Premium ($)')
    axes[idx].set_ylabel('Predicted Premium ($)')
    axes[idx].legend()

plt.suptitle('Predicted vs Actual — Annual Premium', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


## 9.2 Residual Analysis


In [ ]:
# Best overall model
best_split = max(best_results, key=lambda s: max(m['test_r2'] for m in best_results[s].values()))
best_mn = max(best_results[best_split], key=lambda m: best_results[best_split][m]['test_r2'])
best_model_info = best_results[best_split][best_mn]

X_tr, X_te, _, _ = splits[best_split]
_, _, _, y_te_orig = splits_y_orig[best_split]

y_pred_log = best_model_info['model'].predict(X_te)
y_pred = np.expm1(y_pred_log)
residuals = y_te_orig - y_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(residuals, kde=True, bins=50, color='steelblue', ax=axes[0])
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_title('Residual Distribution', fontsize=13)
axes[0].set_xlabel('Residual (Actual - Predicted)')

axes[1].scatter(y_pred, residuals, alpha=0.2, s=5, color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Predicted Values', fontsize=13)
axes[1].set_xlabel('Predicted ($)')
axes[1].set_ylabel('Residual ($)')

plt.suptitle(f'Residual Analysis: {best_mn} ({best_split})', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean residual:   ${residuals.mean():.4f}")
print(f"Std residual:    ${residuals.std():.4f}")


## 9.3 Feature Importance


In [ ]:
# Feature importance from the best tree-based model
if hasattr(best_model_info['model'], 'feature_importances_'):
    importances = best_model_info['model'].feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

    plt.figure(figsize=(12, max(8, len(feat_imp) * 0.2)))
    feat_imp.tail(25).plot(kind='barh', color='steelblue')
    plt.title(f'Top 25 Feature Importances ({best_mn})', fontsize=14)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print("\nTop 15 most important features:")
    print(feat_imp.sort_values(ascending=False).head(15).to_string())
else:
    print(f"{best_mn} does not provide feature importances.")


# 10. Save Best Model


In [ ]:
# Identify and save the overall best model
best_split = max(best_results, key=lambda s: max(m['test_r2'] for m in best_results[s].values()))
best_mn = max(best_results[best_split], key=lambda m: best_results[best_split][m]['test_r2'])
best_model = best_results[best_split][best_mn]['model']
best_r2 = best_results[best_split][best_mn]['test_r2']

model_path = 'best_model_hardrev.pkl'
joblib.dump(best_model, model_path)
print(f"Best model saved: {best_mn}")
print(f"  Split:    {best_split}")
print(f"  Test R2:  {best_r2:.4f}")
print(f"  Test MAE: ${best_results[best_split][best_mn]['test_mae']:.2f}")
print(f"  Test RMSE:${best_results[best_split][best_mn]['test_rmse']:.2f}")
print(f"  Params:   {best_results[best_split][best_mn]['params']}")
print(f"  Saved to: {model_path}")
